**Chapter 19 – Training and Deploying TensorFlow Models at Scale**

_This notebook contains all the sample code and solutions to the exercises in chapter 19._

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-ml3/blob/main/19_training_and_deploying_at_scale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-ml3/blob/main/19_training_and_deploying_at_scale.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Setup

This project requires Python 3.7 or above:

In [ ]:
import sys  # 导入sys模块，用于访问Python版本信息

assert sys.version_info >= (3, 7)  # 断言Python版本至少为3.7，否则抛出异常

**Warning**: the latest TensorFlow versions are based on Keras 3. For chapters 10-15, it wasn't too hard to update the code to support Keras 3, but unfortunately it's much harder for this chapter, so I've had to revert to Keras 2. To do that, I set the `TF_USE_LEGACY_KERAS` environment variable to `"1"` and import the `tf_keras` package. This ensures that `tf.keras` points to `tf_keras`, which is Keras 2.*.

In [ ]:
import os  # 导入os模块，用于设置环境变量
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # 设置环境变量，强制使用Keras 2（旧版本）
import tf_keras  # 导入tf_keras包，确保tf.keras指向Keras 2.*

And TensorFlow ≥ 2.8:

In [ ]:
from packaging import version  # 导入version模块，用于版本号比较
import tensorflow as tf  # 导入TensorFlow库

assert version.parse(tf.__version__) >= version.parse("2.8.0")  # 断言TensorFlow版本至少为2.8.0

If running on Colab or Kaggle, you need to install the Google AI Platform client library, which will be used later in this notebook. You can ignore the warnings about version incompatibilities.

* **Warning**: On Colab, you must restart the Runtime after the installation, and continue with the next cells.

In [ ]:
import sys  # 导入sys模块
if "google.colab" in sys.modules or "kaggle_secrets" in sys.modules:  # 如果在Colab或Kaggle环境中运行
    %pip install -q -U google-cloud-aiplatform  # 安装/升级Google AI Platform客户端库

This chapter discusses how to run or train a model on one or more GPUs, so let's make sure there's at least one, or else issue a warning:

In [ ]:
if not tf.config.list_physical_devices('GPU'):  # 如果没有检测到GPU设备
    print("No GPU was detected. Neural nets can be very slow without a GPU.")  # 打印警告：没有GPU，神经网络会很慢
    if "google.colab" in sys.modules:  # 如果在Colab环境
        print("Go to Runtime > Change runtime and select a GPU hardware "  # 提示用户在Colab中选择GPU
              "accelerator.")
    if "kaggle_secrets" in sys.modules:  # 如果在Kaggle环境
        print("Go to Settings > Accelerator and select GPU.")  # 提示用户在Kaggle中选择GPU

# Serving a TensorFlow Model

Let's start by deploying a model using TF Serving, then we'll deploy to Google Vertex AI.

## Using TensorFlow Serving

The first thing we need to do is to build and train a model, and export it to the SavedModel format.

### Exporting SavedModels

Let's load the MNIST dataset, scale it, and split it.

In [ ]:
from pathlib import Path  # 导入Path类，用于文件路径操作
import tensorflow as tf  # 导入TensorFlow

# 加载并拆分MNIST数据集
mnist = tf.keras.datasets.mnist.load_data()  # 加载MNIST手写数字数据集
(X_train_full, y_train_full), (X_test, y_test) = mnist  # 解包为完整训练集和测试集
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]  # 前5000个样本作为验证集，其余作为训练集
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]  # 对应的标签也做相同拆分

# 构建并训练MNIST模型（同时处理图像预处理）
tf.random.set_seed(42)  # 设置随机种子，保证可重现性
tf.keras.backend.clear_session()  # 清除之前的Keras会话，释放内存
model = tf.keras.Sequential([  # 构建顺序模型
    tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8),  # 展平层：将28x28图像展平为784维向量，输入类型为uint8
    tf.keras.layers.Rescaling(scale=1 / 255),  # 缩放层：将像素值从[0,255]缩放到[0,1]
    tf.keras.layers.Dense(100, activation="relu"),  # 全连接层：100个神经元，ReLU激活函数
    tf.keras.layers.Dense(10, activation="softmax")  # 输出层：10个类别（数字0-9），softmax激活函数
])
model.compile(loss="sparse_categorical_crossentropy",  # 编译模型：使用稀疏分类交叉熵损失函数
              optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),  # 使用SGD优化器，学习率0.01
              metrics=["accuracy"])  # 监控指标为准确率
model.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid))  # 训练模型10个epoch，使用验证集评估

model_name = "my_mnist_model"  # 定义模型名称
model_version = "0001"  # 定义模型版本号
model_path = Path(model_name) / model_version  # 构建模型保存路径：my_mnist_model/0001
model.save(model_path, save_format="tf")  # 以SavedModel格式（TF原生格式）保存模型

Epoch 1/10
1719/1719 [==============================] - 2s 1ms/step - loss: 0.7012 - accuracy: 0.8241 - val_loss: 0.3715 - val_accuracy: 0.9024
Epoch 2/10
1719/1719 [==============================] - 2s 943us/step - loss: 0.3536 - accuracy: 0.9020 - val_loss: 0.2990 - val_accuracy: 0.9144
Epoch 3/10
1719/1719 [==============================] - 2s 933us/step - loss: 0.3036 - accuracy: 0.9145 - val_loss: 0.2651 - val_accuracy: 0.9272
Epoch 4/10
1719/1719 [==============================] - 2s 965us/step - loss: 0.2736 - accuracy: 0.9231 - val_loss: 0.2436 - val_accuracy: 0.9334
Epoch 5/10
1719/1719 [==============================] - 2s 946us/step - loss: 0.2509 - accuracy: 0.9296 - val_loss: 0.2257 - val_accuracy: 0.9364
Epoch 6/10
1719/1719 [==============================] - 2s 974us/step - loss: 0.2322 - accuracy: 0.9350 - val_loss: 0.2121 - val_accuracy: 0.9396
Epoch 7/10
1719/1719 [==============================] - 2s 959us/step - loss: 0.2161 - accuracy: 0.9400 - val_loss: 0.1970 - v

Let's take a look at the file tree (we've discussed what each of these file is used for in chapter 10):

In [ ]:
sorted([str(path) for path in model_path.parent.glob("**/*")])  # 列出模型目录下所有文件和子目录，按字母排序显示

['my_mnist_model/0001',
 'my_mnist_model/0001/assets',
 'my_mnist_model/0001/keras_metadata.pb',
 'my_mnist_model/0001/saved_model.pb',
 'my_mnist_model/0001/variables',
 'my_mnist_model/0001/variables/variables.data-00000-of-00001',
 'my_mnist_model/0001/variables/variables.index']

Let's inspect the SavedModel:

In [ ]:
!saved_model_cli show --dir '{model_path}'  # 使用saved_model_cli命令行工具检查SavedModel的基本信息（标签集）

The given SavedModel contains the following tag-sets:
'serve'


In [ ]:
!saved_model_cli show --dir '{model_path}' --tag_set serve  # 查看"serve"标签集下的签名定义列表

The given SavedModel MetaGraphDef contains SignatureDefs with the following keys:
SignatureDef key: "__saved_model_init_op"
SignatureDef key: "serving_default"


In [ ]:
!saved_model_cli show --dir '{model_path}' --tag_set serve \
                      --signature_def serving_default  # 查看serving_default签名的详细信息（输入输出张量的名称、形状和类型）

The given SavedModel SignatureDef contains the following input(s):
  inputs['flatten_input'] tensor_info:
      dtype: DT_UINT8
      shape: (-1, 28, 28)
      name: serving_default_flatten_input:0
The given SavedModel SignatureDef contains the following output(s):
  outputs['dense_1'] tensor_info:
      dtype: DT_FLOAT
      shape: (-1, 10)
      name: StatefulPartitionedCall:0
Method name is: tensorflow/serving/predict


For even more details, you can run the following command:

```ipython
!saved_model_cli show --dir '{model_path}' --all
```

### Installing and Starting TensorFlow Serving

If you are running this notebook in Colab or Kaggle, TensorFlow Server needs to be installed:

In [ ]:
if "google.colab" in sys.modules or "kaggle_secrets" in sys.modules:  # 如果在Colab或Kaggle环境中
    url = "https://storage.googleapis.com/tensorflow-serving-apt"  # TF Serving的APT仓库地址
    src = "stable tensorflow-model-server tensorflow-model-server-universal"  # APT源的组件
    !echo 'deb {url} {src}' > /etc/apt/sources.list.d/tensorflow-serving.list  # 添加TF Serving的APT源
    !curl '{url}/tensorflow-serving.release.pub.gpg' | apt-key add -  # 添加GPG公钥以验证包的签名
    !apt update -q && apt-get install -y tensorflow-model-server  # 更新包列表并安装TF Serving
    %pip install -q -U tensorflow-serving-api  # 安装TF Serving的Python客户端API

If `tensorflow_model_server` is installed (e.g., if you are running this notebook in Colab), then the following 2 cells will start the server. If your OS is Windows, you may need to run the `tensorflow_model_server` command in a terminal, and replace `${MODEL_DIR}` with the full path to the `my_mnist_model` directory.

In [ ]:
import os  # 导入os模块

os.environ["MODEL_DIR"] = str(model_path.parent.absolute())  # 将模型的父目录绝对路径设为环境变量MODEL_DIR，供TF Serving使用

In [ ]:
%%bash --bg
# 在后台启动TensorFlow Serving服务器
tensorflow_model_server \
    --port=8500 \
    --rest_api_port=8501 \
    --model_name=my_mnist_model \
    --model_base_path="${MODEL_DIR}" >my_server.log 2>&1
# --port=8500: gRPC API端口
# --rest_api_port=8501: REST API端口
# --model_name: 模型名称
# --model_base_path: 模型所在的基础目录
# 日志输出重定向到my_server.log文件

In [ ]:
import time  # 导入time模块

time.sleep(2)  # 等待2秒，让TF Serving服务器有时间启动完成

If you are running this notebook on your own machine, and you prefer to install TF Serving using Docker, first make sure [Docker](https://docs.docker.com/install/) is installed, then run the following commands in a terminal. You must replace `/path/to/my_mnist_model` with the appropriate absolute path to the `my_mnist_model` directory, but do not modify the container path `/models/my_mnist_model`.

```bash
docker pull tensorflow/serving  # downloads the latest TF Serving image

docker run -it --rm -v "/path/to/my_mnist_model:/models/my_mnist_model" \
    -p 8500:8500 -p 8501:8501 -e MODEL_NAME=my_mnist_model tensorflow/serving
```

### Querying TF Serving through the REST API

Next, let's send a REST query to TF Serving:

In [ ]:
import json  # 导入json模块，用于JSON序列化

X_new = X_test[:3]  # 取测试集的前3张图片，假装这是3张新的待分类数字图像
request_json = json.dumps({  # 将请求数据序列化为JSON字符串
    "signature_name": "serving_default",  # 指定使用的签名名称
    "instances": X_new.tolist(),  # 将NumPy数组转为Python列表作为输入实例
})

In [ ]:
request_json[:100] + "..." + request_json[-10:]  # 预览JSON请求的前100个和后10个字符（因为完整内容太长）

'{"signature_name": "serving_default", "instances": [[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..., 0, 0]]]}'

Now let's use TensorFlow Serving's REST API to make predictions:

In [ ]:
import requests  # 导入requests库，用于发送HTTP请求

server_url = "http://localhost:8501/v1/models/my_mnist_model:predict"  # TF Serving REST API的预测端点URL
response = requests.post(server_url, data=request_json)  # 发送POST请求，将JSON数据发送给TF Serving
response.raise_for_status()  # 如果HTTP状态码表示错误，则抛出异常
response = response.json()  # 将响应解析为JSON（Python字典）

In [ ]:
import numpy as np  # 导入NumPy库

y_proba = np.array(response["predictions"])  # 从响应中提取预测概率，转为NumPy数组
y_proba.round(2)  # 四舍五入到小数点后2位，便于查看

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]])

### Querying TF Serving through the gRPC API

In [ ]:
from tensorflow_serving.apis.predict_pb2 import PredictRequest  # 导入gRPC预测请求的protobuf类

request = PredictRequest()  # 创建一个gRPC预测请求对象
request.model_spec.name = model_name  # 设置模型名称
request.model_spec.signature_name = "serving_default"  # 设置签名名称
input_name = model.input_names[0]  # 获取模型的第一个输入名称（== "flatten_input"）
request.inputs[input_name].CopyFrom(tf.make_tensor_proto(X_new))  # 将NumPy数组转为TensorProto格式，并复制到请求的输入中

In [ ]:
import grpc  # 导入gRPC库
from tensorflow_serving.apis import prediction_service_pb2_grpc  # 导入TF Serving的gRPC服务桩

channel = grpc.insecure_channel('localhost:8500')  # 创建一个不安全的gRPC通道（连接到本地8500端口）
predict_service = prediction_service_pb2_grpc.PredictionServiceStub(channel)  # 创建预测服务的客户端桩
response = predict_service.Predict(request, timeout=10.0)  # 发送预测请求，超时时间10秒

Convert the response to a tensor:

In [ ]:
output_name = model.output_names[0]  # 获取模型的第一个输出名称
outputs_proto = response.outputs[output_name]  # 从gRPC响应中获取对应的输出TensorProto
y_proba = tf.make_ndarray(outputs_proto)  # 将TensorProto转换为NumPy数组

In [ ]:
y_proba.round(2)  # 将预测概率四舍五入到2位小数显示

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]],
      dtype=float32)

If your client does not include the TensorFlow library, you can convert the response to a NumPy array like this:

In [ ]:
# 展示如何在不使用tf.make_ndarray()的情况下解析gRPC响应
output_name = model.output_names[0]  # 获取模型输出名称
outputs_proto = response.outputs[output_name]  # 获取输出的TensorProto对象
shape = [dim.size for dim in outputs_proto.tensor_shape.dim]  # 从TensorProto中提取张量形状
y_proba = np.array(outputs_proto.float_val).reshape(shape)  # 从float_val中取出值并重塑为正确形状
y_proba.round(2)  # 四舍五入到2位小数显示

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]])

### Deploying a new model version

In [ ]:
# 构建并训练新版本的MNIST模型
np.random.seed(42)  # 设置NumPy随机种子
tf.random.set_seed(42)  # 设置TensorFlow随机种子
model = tf.keras.Sequential([  # 构建新的顺序模型（结构不同于v1）
    tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8),  # 展平层：将28x28图像展平
    tf.keras.layers.Rescaling(scale=1 / 255),  # 缩放层：归一化像素值到[0,1]
    tf.keras.layers.Dense(50, activation="relu"),  # 第一个全连接层：50个神经元
    tf.keras.layers.Dense(50, activation="relu"),  # 第二个全连接层：50个神经元（比v1多一层但更少神经元）
    tf.keras.layers.Dense(10, activation="softmax")  # 输出层：10个类别
])
model.compile(loss="sparse_categorical_crossentropy",  # 编译模型：稀疏分类交叉熵损失
              optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),  # SGD优化器，学习率0.01
              metrics=["accuracy"])  # 监控准确率
history = model.fit(X_train, y_train, epochs=10,  # 训练10个epoch
                    validation_data=(X_valid, y_valid))  # 使用验证集评估

Epoch 1/10
1719/1719 [==============================] - 2s 931us/step - loss: 0.7039 - accuracy: 0.8056 - val_loss: 0.3418 - val_accuracy: 0.9042
Epoch 2/10
1719/1719 [==============================] - 1s 855us/step - loss: 0.3204 - accuracy: 0.9082 - val_loss: 0.2674 - val_accuracy: 0.9242
Epoch 3/10
1719/1719 [==============================] - 2s 883us/step - loss: 0.2650 - accuracy: 0.9235 - val_loss: 0.2227 - val_accuracy: 0.9368
Epoch 4/10
1719/1719 [==============================] - 1s 869us/step - loss: 0.2319 - accuracy: 0.9329 - val_loss: 0.2032 - val_accuracy: 0.9432
Epoch 5/10
1719/1719 [==============================] - 1s 870us/step - loss: 0.2089 - accuracy: 0.9399 - val_loss: 0.1833 - val_accuracy: 0.9482
Epoch 6/10
1719/1719 [==============================] - 1s 871us/step - loss: 0.1908 - accuracy: 0.9446 - val_loss: 0.1740 - val_accuracy: 0.9498
Epoch 7/10
1719/1719 [==============================] - 2s 873us/step - loss: 0.1756 - accuracy: 0.9490 - val_loss: 0.1605 -

In [ ]:
model_version = "0002"  # 定义新的模型版本号
model_path = Path(model_name) / model_version  # 构建新版本模型的保存路径：my_mnist_model/0002
model.save(model_path, save_format="tf")  # 以SavedModel格式保存新版本模型

INFO:tensorflow:Assets written to: my_mnist_model/0002/assets


Let's take a look at the file tree again:

In [ ]:
sorted([str(path) for path in model_path.parent.glob("**/*")])  # 列出模型目录下所有文件，现在应包含0001和0002两个版本

['my_mnist_model/0001',
 'my_mnist_model/0001/assets',
 'my_mnist_model/0001/keras_metadata.pb',
 'my_mnist_model/0001/saved_model.pb',
 'my_mnist_model/0001/variables',
 'my_mnist_model/0001/variables/variables.data-00000-of-00001',
 'my_mnist_model/0001/variables/variables.index',
 'my_mnist_model/0002',
 'my_mnist_model/0002/assets',
 'my_mnist_model/0002/keras_metadata.pb',
 'my_mnist_model/0002/saved_model.pb',
 'my_mnist_model/0002/variables',
 'my_mnist_model/0002/variables/variables.data-00000-of-00001',
 'my_mnist_model/0002/variables/variables.index']

**Warning**: You may need to wait a minute before the new model is loaded by TensorFlow Serving.

In [ ]:
import requests  # 导入requests库

server_url = "http://localhost:8501/v1/models/my_mnist_model:predict"  # TF Serving REST API预测端点
            
response = requests.post(server_url, data=request_json)  # 发送POST请求（TF Serving会自动使用最新版本模型）
response.raise_for_status()  # 检查HTTP错误
response = response.json()  # 解析JSON响应

In [ ]:
response.keys()  # 查看响应字典的键（应包含"predictions"）

dict_keys(['predictions'])

In [ ]:
y_proba = np.array(response["predictions"])  # 提取预测概率并转为NumPy数组（现在是新模型v2的结果）
y_proba.round(2)  # 四舍五入到2位小数

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ]])

## Creating a Prediction Service on Vertex AI

Follow the instructions in the book to create a Google Cloud Platform account and activate the Vertex AI and Cloud Storage APIs. Then, if you're running this notebook in Colab, you can run the following cell to authenticate using the same Google account as you used with Google Cloud Platform, and authorize this Colab to access your data.

**WARNING: only do this if you trust this notebook!**
* Be extra careful if this is not the official notebook from https://github.com/ageron/handson-ml3: the Colab URL should start with https://colab.research.google.com/github/ageron/handson-ml3. Or else, the code could do whatever it wants with your data.

If you are not running this notebook in Colab, you must follow the instructions in the book to create a service account and generate a key for it, download it to this notebook's directory, and name it `my_service_account_key.json` (or make sure the `GOOGLE_APPLICATION_CREDENTIALS` environment variable points to your key).

In [ ]:
project_id = "my_project"  ##### 请将此处修改为你的GCP项目ID #####

if "google.colab" in sys.modules:  # 如果在Google Colab中运行
    from google.colab import auth  # 导入Colab身份验证模块
    auth.authenticate_user()  # 进行用户身份验证
elif "kaggle_secrets" in sys.modules:  # 如果在Kaggle中运行
    from kaggle_secrets import UserSecretsClient  # 导入Kaggle密钥客户端
    UserSecretsClient().set_gcloud_credentials(project=project_id)  # 设置GCloud凭据
else:  # 如果在本地机器上运行
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "my_service_account_key.json"  # 设置服务账户密钥文件路径

In [ ]:
from google.cloud import storage  # 导入Google Cloud Storage客户端库

bucket_name = "my_bucket"  ##### 请将此处修改为一个唯一的存储桶名称 #####
location = "us-central1"  # 存储桶的地理位置

storage_client = storage.Client(project=project_id)  # 创建GCS客户端实例
bucket = storage_client.create_bucket(bucket_name, location=location)  # 创建新的GCS存储桶
#bucket = storage_client.bucket(bucket_name)  # 如果要复用已有存储桶，取消注释此行

In [ ]:
def upload_directory(bucket, dirpath):  # 定义上传整个目录到GCS的函数
    dirpath = Path(dirpath)  # 将路径转为Path对象
    for filepath in dirpath.glob("**/*"):  # 递归遍历目录下所有文件
        if filepath.is_file():  # 只处理文件（跳过目录）
            blob = bucket.blob(filepath.relative_to(dirpath.parent).as_posix())  # 创建GCS blob，使用相对路径作为blob名称
            blob.upload_from_filename(filepath)  # 上传本地文件到GCS

upload_directory(bucket, "my_mnist_model")  # 将my_mnist_model目录上传到GCS存储桶

In [ ]:
# 多线程版本的upload_directory()函数，速度更快，支持目标路径前缀并打印进度

from concurrent import futures  # 导入futures模块，用于多线程并发

def upload_file(bucket, filepath, blob_path):  # 上传单个文件到GCS
    blob = bucket.blob(blob_path)  # 创建GCS blob对象
    blob.upload_from_filename(filepath)  # 上传文件

def upload_directory(bucket, dirpath, prefix=None, max_workers=50):  # 多线程上传目录
    dirpath = Path(dirpath)  # 转为Path对象
    prefix = prefix or dirpath.name  # 如果没有指定前缀，使用目录名称作为前缀
    with futures.ThreadPoolExecutor(max_workers=max_workers) as executor:  # 创建线程池，最多50个工作线程
        future_to_filepath = {  # 提交所有上传任务到线程池
            executor.submit(  # 提交一个上传任务
                upload_file,
                bucket, filepath,
                f"{prefix}/{filepath.relative_to(dirpath).as_posix()}"  # blob路径 = 前缀/相对路径
            ): filepath
            for filepath in sorted(dirpath.glob("**/*"))  # 递归遍历目录中的所有文件
            if filepath.is_file()  # 只处理文件
        }
        for future in futures.as_completed(future_to_filepath):  # 当每个任务完成时
            filepath = future_to_filepath[future]  # 获取对应的文件路径
            try:
                result = future.result()  # 获取任务结果（如果出错会抛出异常）
            except Exception as ex:
                print(f"Error uploading {filepath!s:60}: {ex}")  # 打印上传错误信息
            else:
                print(f"Uploaded {filepath!s:60}", end="\r")  # 打印上传成功信息（\r覆盖当前行）

    print(f"Uploaded {dirpath!s:60}")  # 打印目录上传完成信息

Alternatively, if you installed Google Cloud CLI (it's preinstalled on Colab), then you can use the following `gsutil` command:

In [ ]:
#!gsutil -m cp -r my_mnist_model gs://{bucket_name}/  # 替代方案：使用gsutil命令行工具多线程递归上传模型目录到GCS

In [ ]:
from google.cloud import aiplatform  # 导入Google Vertex AI客户端库

server_image = "gcr.io/cloud-aiplatform/prediction/tf2-gpu.2-8:latest"  # 指定TF Serving的GPU Docker镜像

aiplatform.init(project=project_id, location=location)  # 初始化Vertex AI客户端，设置项目和区域
mnist_model = aiplatform.Model.upload(  # 将模型上传到Vertex AI模型注册表
    display_name="mnist",  # 模型展示名称
    artifact_uri=f"gs://{bucket_name}/my_mnist_model/0001",  # 模型文件在GCS上的路径（使用v1版本）
    serving_container_image_uri=server_image,  # 用于服务模型的容器镜像
)

Creating Model
Create Model backing LRO: projects/522977795627/locations/us-central1/models/4798114811986575360/operations/53403898236370944
Model created. Resource name: projects/522977795627/locations/us-central1/models/4798114811986575360
To use this Model in another session:
model = aiplatform.Model('projects/522977795627/locations/us-central1/models/4798114811986575360')


**Warning**: this cell may take several minutes to run, as it waits for Vertex AI to provision the compute nodes:

In [ ]:
endpoint = aiplatform.Endpoint.create(display_name="mnist-endpoint")  # 创建一个Vertex AI端点

endpoint.deploy(  # 将模型部署到端点
    mnist_model,  # 要部署的模型
    min_replica_count=1,  # 最小副本数（自动扩缩的下限）
    max_replica_count=5,  # 最大副本数（自动扩缩的上限）
    machine_type="n1-standard-4",  # 计算实例类型：4个vCPU，15GB内存
    accelerator_type="NVIDIA_TESLA_K80",  # GPU类型
    accelerator_count=1  # 每个副本使用1个GPU
)

Creating Endpoint
Create Endpoint backing LRO: projects/522977795627/locations/us-central1/endpoints/5133373499481522176/operations/4135354010494304256
Endpoint created. Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/522977795627/locations/us-central1/endpoints/5133373499481522176')
Deploying Model projects/522977795627/locations/us-central1/models/4798114811986575360 to Endpoint : projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Deploy Endpoint model backing LRO: projects/522977795627/locations/us-central1/endpoints/5133373499481522176/operations/388359120522051584
Endpoint model deployed. Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176


In [ ]:
response = endpoint.predict(instances=X_new.tolist())  # 通过Vertex AI端点进行在线预测

In [ ]:
import numpy as np  # 导入NumPy

np.round(response.predictions, 2)  # 将预测结果四舍五入到2位小数显示

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]])

In [ ]:
endpoint.undeploy_all()  # 取消部署端点上的所有模型
endpoint.delete()  # 删除端点（释放资源，停止计费）

Undeploying Endpoint model: projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Undeploy Endpoint model backing LRO: projects/522977795627/locations/us-central1/endpoints/5133373499481522176/operations/3579722406467469312
Endpoint model undeployed. Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Deleting Endpoint : projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Delete Endpoint  backing LRO: projects/522977795627/locations/us-central1/operations/4738836360561950720
Endpoint deleted. . Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176


## Running Batch Prediction Jobs on Vertex AI

In [ ]:
batch_path = Path("my_mnist_batch")  # 定义批量预测数据的本地目录路径
batch_path.mkdir(exist_ok=True)  # 创建目录（如果已存在则不报错）
with open(batch_path / "my_mnist_batch.jsonl", "w") as jsonl_file:  # 创建JSONL格式的批量输入文件
    for image in X_test[:100].tolist():  # 遍历测试集前100张图片
        jsonl_file.write(json.dumps(image))  # 将每张图片序列化为JSON并写入一行
        jsonl_file.write("\n")  # 每条记录换行（JSONL格式要求）

upload_directory(bucket, batch_path)  # 将批量数据目录上传到GCS

Uploaded my_mnist_batch                                              


In [ ]:
batch_prediction_job = mnist_model.batch_predict(  # 创建并运行批量预测任务
    job_display_name="my_batch_prediction_job",  # 任务显示名称
    machine_type="n1-standard-4",  # 计算实例类型
    starting_replica_count=1,  # 初始副本数
    max_replica_count=5,  # 最大副本数
    accelerator_type="NVIDIA_TESLA_K80",  # GPU类型
    accelerator_count=1,  # 每个副本1个GPU
    gcs_source=[f"gs://{bucket_name}/{batch_path.name}/my_mnist_batch.jsonl"],  # 输入数据在GCS上的路径
    gcs_destination_prefix=f"gs://{bucket_name}/my_mnist_predictions/",  # 预测结果输出到GCS的路径
    sync=True  # 同步执行，等待任务完成（设为False则异步执行）
)

Creating BatchPredictionJob
BatchPredictionJob created. Resource name: projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544
To use this BatchPredictionJob in another session:
bpj = aiplatform.BatchPredictionJob('projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544')
View Batch Prediction Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/batch-predictions/4346926367237996544?project=522977795627
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544 current state:
JobState.JOB_STATE_PENDING
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544 current state:
JobState.JOB_STATE_RUNNING
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544 current state:
JobState.JOB_STATE_RUNNING
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/

In [ ]:
batch_prediction_job.output_info  # 查看批量预测任务的输出信息（输出目录路径等）

gcs_output_directory: "gs://my_bucket/my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z"

In [ ]:
y_probas = []  # 初始化列表，用于存储所有预测概率
for blob in batch_prediction_job.iter_outputs():  # 遍历批量预测任务的所有输出blobs
    print(blob.name)  # 打印blob名称
    if "prediction.results" in blob.name:  # 只处理包含预测结果的文件
        for line in blob.download_as_text().splitlines():  # 下载文件内容并按行分割
            y_proba = json.loads(line)["prediction"]  # 解析JSON获取预测概率
            y_probas.append(y_proba)  # 添加到列表中

my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z/prediction.errors_stats-00000-of-00001
my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z/prediction.results-00000-of-00002
my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z/prediction.results-00001-of-00002


In [ ]:
y_pred = np.argmax(y_probas, axis=1)  # 取每个样本概率最大的类别索引作为预测标签
accuracy = np.sum(y_pred == y_test[:100]) / 100  # 计算前100个测试样本的预测准确率

In [ ]:
accuracy  # 显示批量预测的准确率

0.98

In [ ]:
mnist_model.delete()  # 从Vertex AI模型注册表中删除模型

Deleting Model : projects/522977795627/locations/us-central1/models/4798114811986575360
Delete Model  backing LRO: projects/522977795627/locations/us-central1/operations/598902403101622272
Model deleted. . Resource name: projects/522977795627/locations/us-central1/models/4798114811986575360


Let's delete all the directories we created on GCS (i.e., all the blobs with these prefixes):

In [ ]:
for prefix in ["my_mnist_model/", "my_mnist_batch/", "my_mnist_predictions/"]:  # 遍历需要清理的GCS前缀列表
    blobs = bucket.list_blobs(prefix=prefix)  # 列出该前缀下的所有blob
    for blob in blobs:  # 遍历每个blob
        blob.delete()  # 删除blob

#bucket.delete()  # 如果要删除整个存储桶，取消注释此行
batch_prediction_job.delete()  # 删除批量预测任务记录

Deleting BatchPredictionJob : projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544
Delete BatchPredictionJob  backing LRO: projects/522977795627/locations/us-central1/operations/6699028098374959104
BatchPredictionJob deleted. . Resource name: projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544


# Deploying a Model to a Mobile or Embedded Device

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(str(model_path))  # 从SavedModel创建TFLite转换器
tflite_model = converter.convert()  # 将模型转换为TFLite格式
with open("my_converted_savedmodel.tflite", "wb") as f:  # 以二进制写模式打开文件
    f.write(tflite_model)  # 将TFLite模型写入文件

2022-04-10 09:03:52.237094: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:357] Ignored output_format.
2022-04-10 09:03:52.237108: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:360] Ignored drop_control_dependency.
2022-04-10 09:03:52.237830: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: my_mnist_model/0001
2022-04-10 09:03:52.238869: I tensorflow/cc/saved_model/reader.cc:78] Reading meta graph with tags { serve }
2022-04-10 09:03:52.238881: I tensorflow/cc/saved_model/reader.cc:119] Reading SavedModel debug info (if present) from: my_mnist_model/0001
2022-04-10 09:03:52.242108: I tensorflow/cc/saved_model/loader.cc:228] Restoring SavedModel bundle.
2022-04-10 09:03:52.263868: I tensorflow/cc/saved_model/loader.cc:212] Running initialization op on SavedModel bundle at path: my_mnist_model/0001
2022-04-10 09:03:52.271298: I tensorflow/cc/saved_model/loader.cc:301] SavedModel load for tags { serve }; Status: success: OK. Too

In [ ]:
# 演示如何从Keras模型直接创建TFLite转换器
converter = tf.lite.TFLiteConverter.from_keras_model(model)  # 直接从内存中的Keras模型创建转换器

In [ ]:
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # 启用默认优化（包括训练后量化），可显著减小模型大小

In [ ]:
tflite_model = converter.convert()  # 使用优化后的转换器将模型转换为TFLite格式（包含量化）
with open("my_converted_keras_model.tflite", "wb") as f:  # 以二进制写模式打开文件
    f.write(tflite_model)  # 保存优化后的TFLite模型

INFO:tensorflow:Assets written to: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs/assets


INFO:tensorflow:Assets written to: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs/assets
2022-04-10 09:26:30.319286: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:357] Ignored output_format.
2022-04-10 09:26:30.319301: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:360] Ignored drop_control_dependency.
2022-04-10 09:26:30.319417: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs
2022-04-10 09:26:30.320420: I tensorflow/cc/saved_model/reader.cc:78] Reading meta graph with tags { serve }
2022-04-10 09:26:30.320431: I tensorflow/cc/saved_model/reader.cc:119] Reading SavedModel debug info (if present) from: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs
2022-04-10 09:26:30.323773: I tensorflow/cc/saved_model/loader.cc:228] Restoring SavedModel bundle.
2022-04-10 09:26:30.345416: I tensorflow/cc/saved_model/loader.cc:212] Running initialization

# Running a Model in a Web Page

Code examples for this section are hosted on glitch.com, a website that lets you create Web apps for free.

* https://homl.info/tfjscode: a simple TFJS Web app that loads a pretrained model and classifies an image.
* https://homl.info/tfjswpa: the same Web app setup as a WPA. Try opening this link on various platforms, including mobile devices.
** https://homl.info/wpacode: this WPA's source code.
* https://tensorflow.org/js: The TFJS library.
** https://www.tensorflow.org/js/demos: some fun demos.

# Using GPUs to Speed Up Computations

Let's check that TensorFlow can see the GPU:

In [ ]:
physical_gpus = tf.config.list_physical_devices("GPU")  # 列出所有可用的物理GPU设备
physical_gpus  # 显示GPU列表

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


If you want your TensorFlow script to use only GPUs \#0 and \#1 (based on PCI order), then you can set the environment variables `CUDA_DEVICE_ORDER=PCI_BUS_ID` and `CUDA_VISIBLE_DEVICES=0,1` before starting your script, or in the script itself before using TensorFlow.

## Managing the GPU RAM

To limit the amount of RAM to 2GB per GPU:

In [ ]:
# 限制每个GPU的显存使用为2GB（取消注释以启用）
#for gpu in physical_gpus:  # 遍历每个物理GPU
#    tf.config.set_logical_device_configuration(  # 设置逻辑设备配置
#        gpu,
#        [tf.config.LogicalDeviceConfiguration(memory_limit=2048)]  # 限制显存为2048MB（2GB）
#    )

To make TensorFlow grab memory as it needs it (only releasing it when the process shuts down):

In [ ]:
# 让TensorFlow按需动态分配GPU显存，而非一次性占满（取消注释以启用）
#for gpu in physical_gpus:  # 遍历每个物理GPU
#    tf.config.experimental.set_memory_growth(gpu, True)  # 启用显存增长模式

Equivalently, you can set the `TF_FORCE_GPU_ALLOW_GROWTH` environment variable to `true` before using TensorFlow.

To split a physical GPU into two logical GPUs:

In [ ]:
# 将一个物理GPU拆分为两个逻辑GPU（取消注释以启用）
#tf.config.set_logical_device_configuration(  # 设置逻辑设备配置
#    physical_gpus[0],  # 对第一个物理GPU进行配置
#    [tf.config.LogicalDeviceConfiguration(memory_limit=2048),  # 逻辑GPU 0：2GB显存
#     tf.config.LogicalDeviceConfiguration(memory_limit=2048)]  # 逻辑GPU 1：2GB显存
#)

In [ ]:
logical_gpus = tf.config.list_logical_devices("GPU")  # 列出所有逻辑GPU设备（包括拆分后的虚拟GPU）
logical_gpus  # 显示逻辑GPU列表

[LogicalDevice(name='/device:GPU:0', device_type='GPU')]


## Placing Operations and Variables on Devices

To log every variable and operation placement (this must be run just after importing TensorFlow):

In [ ]:
#tf.get_logger().setLevel("DEBUG")  # 将日志级别设为DEBUG（默认为INFO），显示更详细的日志信息
#tf.debugging.set_log_device_placement(True)  # 启用设备放置日志，打印每个操作和变量被分配到哪个设备

In [ ]:
a = tf.Variable([1., 2., 3.])  # 创建float32类型的变量（如果有GPU，会自动放在GPU上）
a.device  # 查看变量所在的设备

'/job:localhost/replica:0/task:0/device:GPU:0'

In [ ]:
b = tf.Variable([1, 2, 3])  # 创建int32类型的变量（整数变量通常放在CPU上，因为GPU对整数运算支持有限）
b.device  # 查看变量所在的设备

'/job:localhost/replica:0/task:0/device:CPU:0'

You can place variables and operations manually on the desired device using a `tf.device()` context:

In [ ]:
with tf.device("/cpu:0"):  # 使用tf.device()上下文管理器手动指定设备为CPU
    c = tf.Variable([1., 2., 3.])  # 在CPU上创建float32变量

c.device  # 确认变量确实在CPU上

'/job:localhost/replica:0/task:0/device:CPU:0'

If you specify a device that does not exist, or for which there is no kernel, TensorFlow will silently fallback to the default placement:

In [ ]:
# 演示指定不存在的设备时的默认回退行为

with tf.device("/gpu:1234"):  # 指定一个不存在的GPU设备
    d = tf.Variable([1., 2., 3.])  # TensorFlow会静默回退到默认设备

d.device  # 查看变量实际被分配到的设备

"'/job:localhost/replica:0/task:0/device:GPU:0'"

If you want TensorFlow to throw an exception when you try to use a device that does not exist, instead of falling back to the default device:

In [ ]:
tf.config.set_soft_device_placement(False)  # 关闭软设备放置，指定不存在的设备时会抛出异常而非静默回退

# 演示在软设备放置关闭时，使用不存在的设备会引发异常
try:
    with tf.device("/gpu:1000"):  # 尝试使用不存在的GPU
        d = tf.Variable([1., 2., 3.])  # 这将抛出异常
except tf.errors.InvalidArgumentError as ex:  # 捕获无效参数异常
    print(ex)  # 打印异常信息

tf.config.set_soft_device_placement(True)  # 恢复软设备放置模式

Could not satisfy device specification '/job:localhost/replica:0/task:0/device:GPU:1000'. enable_soft_placement=0. Supported device types [CPU]. All available devices [/job:localhost/replica:0/task:0/device:CPU:0].


## Parallel Execution Across Multiple Devices

If you want to set the number of inter-op or intra-op threads (this may be useful if you want to avoid saturating the CPU, or if you want to make TensorFlow single-threaded, to run a perfectly reproducible test case):

In [ ]:
#tf.config.threading.set_inter_op_parallelism_threads(10)  # 设置操作间并行线程数（不同操作之间的并行执行）
#tf.config.threading.set_intra_op_parallelism_threads(10)  # 设置操作内并行线程数（单个操作内部的并行执行）

# Training Models Across Multiple Devices

## Training at Scale Using the Distribution Strategies API

In [ ]:
# 创建一个用于MNIST分类的CNN模型
def create_model():
    return tf.keras.Sequential([  # 构建顺序模型
        tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28],  # 重塑层：将28x28展平输入重塑为28x28x1（添加通道维度）
                                dtype=tf.uint8),
        tf.keras.layers.Rescaling(scale=1 / 255),  # 缩放层：像素值归一化到[0,1]
        tf.keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",  # 第一个卷积层：64个7x7卷积核，ReLU激活
                               padding="same"),  # same填充，保持空间尺寸不变
        tf.keras.layers.MaxPooling2D(pool_size=2),  # 最大池化层：2x2池化，尺寸减半为14x14
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",  # 第二个卷积层：128个3x3卷积核
                               padding="same"), 
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",  # 第三个卷积层：128个3x3卷积核
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),  # 最大池化层：尺寸减半为7x7
        tf.keras.layers.Flatten(),  # 展平层：将多维特征图展平为一维向量
        tf.keras.layers.Dense(units=64, activation="relu"),  # 全连接层：64个神经元
        tf.keras.layers.Dropout(0.5),  # Dropout层：训练时随机丢弃50%的神经元，防止过拟合
        tf.keras.layers.Dense(units=10, activation="softmax"),  # 输出层：10个类别，softmax激活
    ])

In [ ]:
tf.random.set_seed(42)  # 设置随机种子

strategy = tf.distribute.MirroredStrategy()  # 创建镜像分布策略（在所有可用GPU上复制模型）

with strategy.scope():  # 在策略作用域中创建和编译模型
    model = create_model()  # 创建Keras模型（会在每个GPU上创建一个副本）
    model.compile(loss="sparse_categorical_crossentropy",  # 编译模型：稀疏分类交叉熵损失
                  optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),  # SGD优化器
                  metrics=["accuracy"])  # 监控准确率

batch_size = 100  # 批量大小（最好能被副本数量整除，以便均匀分配）
model.fit(X_train, y_train, epochs=10,  # 训练10个epoch
          validation_data=(X_valid, y_valid), batch_size=batch_size)  # 每个批次会自动分配到各个GPU

In [ ]:
type(model.weights[0])  # 查看模型权重的类型（使用MirroredStrategy时，权重类型为MirroredVariable）

tensorflow.python.distribute.values.MirroredVariable

In [ ]:
model.predict(X_new).round(2)  # 使用模型进行预测（批次会自动分配到各个副本），结果四舍五入到2位小数

array([[0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)


In [ ]:
# 演示保存和加载模型不会保留分布策略
model.save("my_mirrored_model", save_format="tf")  # 保存模型为SavedModel格式
model = tf.keras.models.load_model("my_mirrored_model")  # 重新加载模型（不在策略作用域中）
type(model.weights[0])  # 权重类型不再是MirroredVariable，而是普通的ResourceVariable

INFO:tensorflow:Assets written to: my_mirrored_model/assets


tensorflow.python.ops.resource_variable_ops.ResourceVariable

In [ ]:
with strategy.scope():  # 在策略作用域中加载模型，权重会恢复为MirroredVariable
    model = tf.keras.models.load_model("my_mirrored_model")  # 加载模型（在分布策略作用域内）

In [ ]:
type(model.weights[0])  # 确认权重类型现在是MirroredVariable（分布策略已恢复）

tensorflow.python.distribute.values.MirroredVariable


If you want to specify the list of GPUs to use:

In [ ]:
strategy = tf.distribute.MirroredStrategy(devices=["/gpu:0", "/gpu:1"])  # 创建镜像策略，指定只使用GPU 0和GPU 1

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


If you want to change the default all-reduce algorithm:

In [ ]:
strategy = tf.distribute.MirroredStrategy(  # 创建镜像策略，自定义AllReduce算法
    cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())  # 使用分层复制AllReduce（适用于多节点多GPU拓扑）

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


If you want to use the `CentralStorageStrategy`:

In [ ]:
strategy = tf.distribute.experimental.CentralStorageStrategy()  # 创建集中存储策略（参数存储在CPU上，计算在GPU上执行）

INFO:tensorflow:ParameterServerStrategy (CentralStorageStrategy if you are using a single machine) with compute_devices = ['/job:localhost/replica:0/task:0/device:CPU:0'], variable_device = '/job:localhost/replica:0/task:0/device:CPU:0'


In [ ]:
# 在Google Colab上使用TPU进行训练（取消注释以启用）
#if "google.colab" in sys.modules and "COLAB_TPU_ADDR" in os.environ:  # 检查是否在Colab中且TPU可用
#  tpu_address = "grpc://" + os.environ["COLAB_TPU_ADDR"]  # 获取TPU的gRPC地址
#else:
#  tpu_address = ""  # 如果不在Colab或没有TPU，使用空字符串
#resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu_address)  # 创建TPU集群解析器
#tf.config.experimental_connect_to_cluster(resolver)  # 连接到TPU集群
#tf.tpu.experimental.initialize_tpu_system(resolver)  # 初始化TPU系统
#strategy = tf.distribute.experimental.TPUStrategy(resolver)  # 创建TPU分布策略

## Training a Model on a TensorFlow Cluster

A TensorFlow cluster is a group of TensorFlow processes running in parallel, usually on different machines, and talking to each other to complete some work, for example training or executing a neural network. Each TF process in the cluster is called a "task" (or a "TF server"). It has an IP address, a port, and a type (also called its role or its job). The type can be `"worker"`, `"chief"`, `"ps"` (parameter server) or `"evaluator"`:
* Each **worker** performs computations, usually on a machine with one or more GPUs.
* The **chief** performs computations as well, but it also handles extra work such as writing TensorBoard logs or saving checkpoints. There is a single chief in a cluster. If it is not defined, then it is worker #0.
* A **parameter server** (ps) only keeps track of variable values, it is usually on a CPU-only machine.
* The **evaluator** obviously takes care of evaluation. There is usually a single evaluator in a cluster.

The set of tasks that share the same type is often called a "job". For example, the "worker" job is the set of all workers.

To start a TensorFlow cluster, you must first define it. This means specifying all the tasks (IP address, TCP port, and type). For example, the following cluster specification defines a cluster with 3 tasks (2 workers and 1 parameter server). It's a dictionary with one key per job, and the values are lists of task addresses:

In [ ]:
cluster_spec = {  # 定义TensorFlow集群规范（指定所有任务的地址）
    "worker": [  # worker任务列表（负责计算）
        "machine-a.example.com:2222",     # /job:worker/task:0 - 第一个worker的地址
        "machine-b.example.com:2222"      # /job:worker/task:1 - 第二个worker的地址
    ],
    "ps": ["machine-a.example.com:2221"]  # /job:ps/task:0 - 参数服务器的地址
}

Every task in the cluster may communicate with every other task in the server, so make sure to configure your firewall to authorize all communications between these machines on these ports (it's usually simpler if you use the same port on every machine).

When a task is started, it needs to be told which one it is: its type and index (the task index is also called the task id). A common way to specify everything at once (both the cluster spec and the current task's type and id) is to set the `TF_CONFIG` environment variable before starting the program. It must be a JSON-encoded dictionary containing a cluster specification (under the `"cluster"` key), and the type and index of the task to start (under the `"task"` key). For example, the following `TF_CONFIG` environment variable defines the same cluster as above, with 2 workers and 1 parameter server, and specifies that the task to start is worker \#0:

In [ ]:
os.environ["TF_CONFIG"] = json.dumps({  # 设置TF_CONFIG环境变量（JSON格式），用于配置分布式训练
    "cluster": cluster_spec,  # 集群规范（所有任务的地址）
    "task": {"type": "worker", "index": 0}  # 当前任务的类型（worker）和索引（0，即第一个worker）
})

Some platforms (e.g., Google Vertex AI) automatically set this environment variable for you.

TensorFlow's `TFConfigClusterResolver` class reads the cluster configuration from this environment variable:

In [ ]:
resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()  # 从TF_CONFIG环境变量读取集群配置
resolver.cluster_spec()  # 获取集群规范对象（显示所有任务的地址信息）

ClusterSpec({'ps': ['machine-a.example.com:2221'], 'worker': ['machine-a.example.com:2222', 'machine-b.example.com:2222']})

In [ ]:
resolver.task_type  # 获取当前任务的类型（如"worker"、"chief"、"ps"）

'worker'

In [ ]:
resolver.task_id  # 获取当前任务的索引号（从0开始）

0

Now let's run a simpler cluster with just two worker tasks, both running on the local machine. We will use the `MultiWorkerMirroredStrategy` to train a model across these two tasks.

The first step is to write the training code. As this code will be used to run both workers, each in its own process, we write this code to a separate Python file, `my_mnist_multiworker_task.py`. The code is relatively straightforward, but there are a couple important things to note:
* We create the `MultiWorkerMirroredStrategy` before doing anything else with TensorFlow.
* Only one of the workers will take care of logging to TensorBoard. As mentioned earlier, this worker is called the *chief*. When it is not defined explicitly, then by convention it is worker #0.

In [ ]:
%%writefile my_mnist_multiworker_task.py
# 将以下内容写入my_mnist_multiworker_task.py文件（多Worker分布式训练脚本）

import tempfile  # 导入临时目录模块
import tensorflow as tf  # 导入TensorFlow

strategy = tf.distribute.MultiWorkerMirroredStrategy()  # 在最开始创建多Worker镜像策略！（必须在使用TF之前）
resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()  # 从TF_CONFIG读取集群配置
print(f"Starting task {resolver.task_type} #{resolver.task_id}")  # 打印当前任务类型和ID

# 加载并拆分MNIST数据集
mnist = tf.keras.datasets.mnist.load_data()  # 加载MNIST数据集
(X_train_full, y_train_full), (X_test, y_test) = mnist  # 解包训练集和测试集
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]  # 拆分验证集和训练集
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]  # 对应标签

with strategy.scope():  # 在策略作用域中创建和编译模型
    model = tf.keras.Sequential([  # 构建CNN模型
        tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28],  # 重塑为带通道维度的图像
                                dtype=tf.uint8),
        tf.keras.layers.Rescaling(scale=1 / 255),  # 像素值归一化
        tf.keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",  # 第一个卷积层
                               padding="same", input_shape=[28, 28, 1]),
        tf.keras.layers.MaxPooling2D(pool_size=2),  # 最大池化
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",  # 第二个卷积层
                               padding="same"), 
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",  # 第三个卷积层
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),  # 最大池化
        tf.keras.layers.Flatten(),  # 展平
        tf.keras.layers.Dense(units=64, activation="relu"),  # 全连接层
        tf.keras.layers.Dropout(0.5),  # Dropout防过拟合
        tf.keras.layers.Dense(units=10, activation="softmax"),  # 输出层
    ])
    model.compile(loss="sparse_categorical_crossentropy",  # 编译模型
                  optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])

model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=10)  # 训练模型

if resolver.task_id == 0:  # 如果是chief（worker #0），保存模型到正确位置
    model.save("my_mnist_multiworker_model", save_format="tf")
else:
    tmpdir = tempfile.mkdtemp()  # 其他worker保存到临时目录
    model.save(tmpdir, save_format="tf")  # 保存模型（TF要求所有worker都执行保存操作）
    tf.io.gfile.rmtree(tmpdir)  # 训练完成后删除临时目录

Writing my_mnist_multiworker_task.py


In a real world application, there would typically be a single worker per machine, but in this example we're running both workers on the same machine, so they will both try to use all the available GPU RAM (if this machine has a GPU), and this will likely lead to an Out-Of-Memory (OOM) error. To avoid this, we could use the `CUDA_VISIBLE_DEVICES` environment variable to assign a different GPU to each worker. Alternatively, we can simply disable GPU support, by setting `CUDA_VISIBLE_DEVICES` to an empty string.

We are now ready to start both workers, each in its own process. Notice that we change the task index:

In [ ]:
%%bash --bg
# 在后台启动Worker 0（chief）
export CUDA_VISIBLE_DEVICES=''  # 禁用GPU（两个worker在同一台机器上，避免GPU显存冲突）
export TF_CONFIG='{"cluster": {"worker": ["127.0.0.1:9901", "127.0.0.1:9902"]},
                   "task": {"type": "worker", "index": 0}}'  # 设置Worker 0的TF_CONFIG
python my_mnist_multiworker_task.py > my_worker_0.log 2>&1  # 运行训练脚本，日志输出到文件

In [ ]:
%%bash --bg
# 在后台启动Worker 1
export CUDA_VISIBLE_DEVICES=''  # 禁用GPU
export TF_CONFIG='{"cluster": {"worker": ["127.0.0.1:9901", "127.0.0.1:9902"]},
                   "task": {"type": "worker", "index": 1}}'  # 设置Worker 1的TF_CONFIG（注意index改为1）
python my_mnist_multiworker_task.py > my_worker_1.log 2>&1  # 运行同一个训练脚本，日志输出到另一个文件

**Note**: if you get warnings about `AutoShardPolicy`, you can safely ignore them. See [TF issue #42146](https://github.com/tensorflow/tensorflow/issues/42146) for more details.

That's it! Our TensorFlow cluster is now running, but we can't see it in this notebook because it's running in separate processes (but you can see the progress in `my_worker_*.log`).

Since the chief (worker #0) is writing to TensorBoard, we use TensorBoard to view the training progress. Run the following cell, then click on the settings button (i.e., the gear icon) in the TensorBoard interface and check the "Reload data" box to make TensorBoard automatically refresh every 30s. Once the first epoch of training is finished (which may take a few minutes), and once TensorBoard refreshes, the SCALARS tab will appear. Click on this tab to view the progress of the model's training and validation accuracy.

In [ ]:
%load_ext tensorboard  # 加载TensorBoard Jupyter扩展
%tensorboard --logdir=./my_mnist_multiworker_logs --port=6006  # 启动TensorBoard，指向多Worker训练日志目录

In [ ]:
# 使用NCCL（NVIDIA集合通信库）作为通信后端的多Worker镜像策略（取消注释以启用）
# strategy = tf.distribute.MultiWorkerMirroredStrategy(
#     communication_options=tf.distribute.experimental.CommunicationOptions(
#         implementation=tf.distribute.experimental.CollectiveCommunication.NCCL))  # NCCL通常在多GPU环境中性能更好

## Running Large Training Jobs on Vertex AI

Let's copy the training script, but add `import os` and change the save path to be the GCS path that the `AIP_MODEL_DIR` environment variable will point to:

In [ ]:
%%writefile my_vertex_ai_training_task.py
# 将以下内容写入Vertex AI训练脚本文件

import os  # 导入os模块
from pathlib import Path  # 导入Path类
import tempfile  # 导入临时目录模块
import tensorflow as tf  # 导入TensorFlow

strategy = tf.distribute.MultiWorkerMirroredStrategy()  # 在最开始创建多Worker镜像策略
resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()  # 从TF_CONFIG读取集群配置

if resolver.task_type == "chief":  # 如果当前任务是chief节点
    model_dir = os.getenv("AIP_MODEL_DIR")  # 从Vertex AI环境变量获取模型保存路径
    tensorboard_log_dir = os.getenv("AIP_TENSORBOARD_LOG_DIR")  # 获取TensorBoard日志目录
    checkpoint_dir = os.getenv("AIP_CHECKPOINT_DIR")  # 获取检查点目录
else:  # 其他worker使用临时目录
    tmp_dir = Path(tempfile.mkdtemp())  # 创建临时目录
    model_dir = tmp_dir / "model"  # 临时模型目录
    tensorboard_log_dir = tmp_dir / "logs"  # 临时日志目录
    checkpoint_dir = tmp_dir / "ckpt"  # 临时检查点目录

callbacks = [tf.keras.callbacks.TensorBoard(tensorboard_log_dir),  # TensorBoard回调
             tf.keras.callbacks.ModelCheckpoint(checkpoint_dir)]  # 模型检查点回调

# 加载并准备MNIST数据集
mnist = tf.keras.datasets.mnist.load_data()  # 加载MNIST数据集
(X_train_full, y_train_full), (X_test, y_test) = mnist  # 解包
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]  # 拆分验证集
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]  # 拆分标签

# 在分布策略作用域中构建和编译Keras模型
with strategy.scope():
    model = tf.keras.Sequential([  # 构建CNN模型
        tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28],  # 重塑输入
                                dtype=tf.uint8),
        tf.keras.layers.Lambda(lambda X: X / 255),  # Lambda层进行归一化
        tf.keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",  # 卷积层
                               padding="same", input_shape=[28, 28, 1]),
        tf.keras.layers.MaxPooling2D(pool_size=2),  # 池化层
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",  # 卷积层
                               padding="same"), 
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",  # 卷积层
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),  # 池化层
        tf.keras.layers.Flatten(),  # 展平层
        tf.keras.layers.Dense(units=64, activation="relu"),  # 全连接层
        tf.keras.layers.Dropout(0.5),  # Dropout层
        tf.keras.layers.Dense(units=10, activation="softmax"),  # 输出层
    ])
    model.compile(loss="sparse_categorical_crossentropy",  # 编译模型
                  optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])

model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=10,  # 训练模型
          callbacks=callbacks)  # 使用TensorBoard和CheckPoint回调
model.save(model_dir, save_format="tf")  # 保存模型到指定目录

Writing my_vertex_ai_training_task.py


In [ ]:
custom_training_job = aiplatform.CustomTrainingJob(  # 创建Vertex AI自定义训练任务
    display_name="my_custom_training_job",  # 任务显示名称
    script_path="my_vertex_ai_training_task.py",  # 训练脚本路径
    container_uri="gcr.io/cloud-aiplatform/training/tf-gpu.2-4:latest",  # 训练用的Docker镜像
    model_serving_container_image_uri=server_image,  # 模型服务用的Docker镜像
    requirements=["gcsfs==2022.3.0"],  # 额外的Python依赖（此处仅作示例）
    staging_bucket=f"gs://{bucket_name}/staging"  # 暂存文件的GCS存储桶路径
)

In [ ]:
mnist_model2 = custom_training_job.run(  # 运行自定义训练任务
    machine_type="n1-standard-4",  # 计算实例类型
    replica_count=2,  # 副本数（2个worker）
    accelerator_type="NVIDIA_TESLA_K80",  # GPU类型
    accelerator_count=2,  # 每个副本使用2个GPU
)

Training script copied to:
gs://my_bucket/aiplatform-2022-04-14-10:08:24.124-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://my_bucket/aiplatform-custom-training-2022-04-14-10:08:25.226 
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5407999068506947584?project=522977795627
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/5407999068506947584 current state:
PipelineState.PIPELINE_STATE_PENDING
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/5407999068506947584 current state:
PipelineState.PIPELINE_STATE_RUNNING
View backing custom job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/6685701948726837248?project=522977795627
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/5407999068506947584 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/522977795627/locations/us-c

Let's clean up:

In [ ]:
mnist_model2.delete()  # 删除Vertex AI中的模型
custom_training_job.delete()  # 删除自定义训练任务
blobs = bucket.list_blobs(prefix=f"gs://{bucket_name}/staging/")  # 列出暂存目录中的所有blob
for blob in blobs:  # 遍历每个blob
    blob.delete()  # 删除blob（清理暂存文件）

# Hyperparameter Tuning on Vertex AI

In [ ]:
%%writefile my_vertex_ai_trial.py
# 将以下内容写入Vertex AI超参数调优试验脚本

import argparse  # 导入命令行参数解析模块

parser = argparse.ArgumentParser()  # 创建参数解析器
parser.add_argument("--n_hidden", type=int, default=2)  # 隐藏层数量参数
parser.add_argument("--n_neurons", type=int, default=256)  # 每层神经元数量参数
parser.add_argument("--learning_rate", type=float, default=1e-2)  # 学习率参数
parser.add_argument("--optimizer", default="adam")  # 优化器类型参数
args = parser.parse_args()  # 解析命令行参数

import tensorflow as tf  # 导入TensorFlow

def build_model(args):  # 根据超参数构建模型
    with tf.distribute.MirroredStrategy().scope():  # 使用镜像策略（利用所有可用GPU）
        model = tf.keras.Sequential()  # 构建顺序模型
        model.add(tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8))  # 展平层
        for _ in range(args.n_hidden):  # 根据n_hidden参数添加隐藏层
            model.add(tf.keras.layers.Dense(args.n_neurons, activation="relu"))  # 全连接层
        model.add(tf.keras.layers.Dense(10, activation="softmax"))  # 输出层
        opt = tf.keras.optimizers.get(args.optimizer)  # 根据名称获取优化器实例
        opt.learning_rate = args.learning_rate  # 设置学习率
        model.compile(loss="sparse_categorical_crossentropy", optimizer=opt,  # 编译模型
                      metrics=["accuracy"])
        return model

# 加载并拆分数据集
mnist = tf.keras.datasets.mnist.load_data()  # 加载MNIST
(X_train_full, y_train_full), (X_test, y_test) = mnist  # 解包
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]  # 拆分验证集
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]  # 拆分标签

# 使用Vertex AI提供的环境变量并创建回调
import os  # 导入os模块
model_dir = os.getenv("AIP_MODEL_DIR")  # 模型保存路径
tensorboard_log_dir = os.getenv("AIP_TENSORBOARD_LOG_DIR")  # TensorBoard日志路径
checkpoint_dir = os.getenv("AIP_CHECKPOINT_DIR")  # 检查点路径
trial_id = os.getenv("CLOUD_ML_TRIAL_ID")  # 当前试验ID
tensorboard_cb = tf.keras.callbacks.TensorBoard(tensorboard_log_dir)  # TensorBoard回调
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=5)  # 早停回调（5个epoch无改善则停止）
callbacks = [tensorboard_cb, early_stopping_cb]  # 回调列表

model = build_model(args)  # 使用命令行参数构建模型
history = model.fit(X_train, y_train, validation_data=(X_valid, y_valid),  # 训练模型
                    epochs=10, callbacks=callbacks)
model.save(model_dir, save_format="tf")  # 保存模型

import hypertune  # 导入Google Cloud超参数调优库

hypertune = hypertune.HyperTune()  # 创建HyperTune实例
hypertune.report_hyperparameter_tuning_metric(  # 向Vertex AI报告指标
    hyperparameter_metric_tag="accuracy",  # 报告的指标名称
    metric_value=max(history.history["val_accuracy"]),  # 报告最大验证准确率
    global_step=model.optimizer.iterations.numpy(),  # 当前全局步数
)

Writing my_vertex_ai_trial.py


In [ ]:
trial_job = aiplatform.CustomJob.from_local_script(  # 从本地脚本创建Vertex AI自定义作业
    display_name="my_search_trial_job",  # 作业显示名称
    script_path="my_vertex_ai_trial.py",  # 训练脚本路径
    container_uri="gcr.io/cloud-aiplatform/training/tf-gpu.2-4:latest",  # 训练容器镜像
    staging_bucket=f"gs://{bucket_name}/staging",  # 暂存存储桶
    accelerator_type="NVIDIA_TESLA_K80",  # GPU类型
    accelerator_count=2,  # 每个试验使用2个GPU
)

Training script copied to:
gs://homl3-mybucket5/staging/aiplatform-2022-04-18-18:14:02.860-aiplatform_custom_trainer_script-0.1.tar.gz.


In [ ]:
from google.cloud.aiplatform import hyperparameter_tuning as hpt  # 导入超参数调优模块

hp_job = aiplatform.HyperparameterTuningJob(  # 创建超参数调优任务
    display_name="my_hp_search_job",  # 任务显示名称
    custom_job=trial_job,  # 使用之前定义的自定义作业作为试验模板
    metric_spec={"accuracy": "maximize"},  # 优化目标：最大化准确率
    parameter_spec={  # 超参数搜索空间定义
        "learning_rate": hpt.DoubleParameterSpec(min=1e-3, max=10, scale="log"),  # 学习率：对数尺度的浮点数
        "n_neurons": hpt.IntegerParameterSpec(min=1, max=300, scale="linear"),  # 神经元数：线性尺度的整数
        "n_hidden": hpt.IntegerParameterSpec(min=1, max=10, scale="linear"),  # 隐藏层数：线性尺度的整数
        "optimizer": hpt.CategoricalParameterSpec(["sgd", "adam"]),  # 优化器：分类参数
    },
    max_trial_count=100,  # 最多运行100个试验
    parallel_trial_count=20,  # 最多20个试验并行运行
)
hp_job.run()  # 运行超参数调优任务

Creating HyperparameterTuningJob
HyperparameterTuningJob created. Resource name: projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568
To use this HyperparameterTuningJob in another session:
hpt_job = aiplatform.HyperparameterTuningJob.get('projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568')
View HyperparameterTuningJob:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5825136187899117568?project=522977795627
HyperparameterTuningJob projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJob projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJob projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJ

In [ ]:
def get_final_metric(trial, metric_id):  # 定义函数：从试验结果中提取指定指标的最终值
    for metric in trial.final_measurement.metrics:  # 遍历试验的所有最终指标
        if metric.metric_id == metric_id:  # 找到匹配的指标
            return metric.value  # 返回指标值

trials = hp_job.trials  # 获取所有试验结果
trial_accuracies = [get_final_metric(trial, "accuracy") for trial in trials]  # 提取每个试验的准确率
best_trial = trials[np.argmax(trial_accuracies)]  # 找到准确率最高的试验

In [ ]:
max(trial_accuracies)  # 显示所有试验中的最高准确率

0.977400004863739

In [ ]:
best_trial.id  # 显示最佳试验的ID

'98'

In [ ]:
best_trial.parameters  # 显示最佳试验使用的超参数组合

[parameter_id: "learning_rate"
value {
  number_value: 0.001
}
, parameter_id: "n_hidden"
value {
  number_value: 8.0
}
, parameter_id: "n_neurons"
value {
  number_value: 216.0
}
, parameter_id: "optimizer"
value {
  string_value: "adam"
}
]

# Extra Material – Distributed Keras Tuner on Vertex AI

Instead of using Vertex AI's hyperparameter tuning service, you can use [Keras Tuner](https://keras.io/keras_tuner/) (introduced in Chapter 10) and run it on Vertex AI VMs. Keras Tuner provides a simple way to scale hyperparameter search by distributing it across multiple machines: it only requires setting three environment variables on each machine, then running your regular Keras Tuner code on each machine. You can use the exact same script on all machines. One of the machines acts as the chief, and the others act as workers. Each worker asks the chief which hyperparameter values to try—it acts as the oracle—then the worker trains the model using these hyperparameter values, and finally it reports the model's performance back to the chief, which can then decide which hyperparameter values the worker should try next.

The three environment variables you need to set on each machine are:

* `KERASTUNER_TUNER_ID`: equal to `"chief"` on the chief machine, or a unique identifier on each worker machine, such as `"worker0"`, `"worker1"`, etc.
* `KERASTUNER_ORACLE_IP`: the IP address or hostname of the chief machine. The chief itself should generally use `"0.0.0.0"` to listen on every IP address on the machine.
* `KERASTUNER_ORACLE_PORT`: the TCP port that the chief will be listening on.

You can use distributed Keras Tuner on any set of machines. If you want to run it on Vertex AI machines, then you can spawn a regular training job, and just modify the training script to set the three environment variables properly before using Keras Tuner.

For example, the script below starts by parsing the `TF_CONFIG` environment variable, which will be automatically set by Vertex AI, just like earlier. It finds the address of the task of type `"chief"`, and it extracts the IP address or hostname, and the TCP port. It then defines the tuner ID as the task type followed by the task index, for example `"worker0"`. If the tuner ID is `"chief0"`, it changes it to `"chief"`, and it sets the IP to `"0.0.0.0"`: this will make it listen on all IPv4 address on its machine. Then it defines the environment variables for Keras Tuner. Next, the script creates a tuner, just like in Chapter 10, the it runs the search, and finally it saves the best model to the location given by Vertex AI:

In [ ]:
%%writefile my_keras_tuner_search.py
# 将以下内容写入Keras Tuner分布式搜索脚本

import json  # 导入json模块
import os  # 导入os模块

tf_config = json.loads(os.environ["TF_CONFIG"])  # 解析Vertex AI设置的TF_CONFIG环境变量

chief_ip, chief_port = tf_config["cluster"]["chief"][0].rsplit(":", 1)  # 从集群配置中提取chief的IP和端口
tuner_id = f'{tf_config["task"]["type"]}{tf_config["task"]["index"]}'  # 构造tuner ID，如"chief0"或"worker0"
if tuner_id == "chief0":  # 如果当前是chief节点
    tuner_id = "chief"  # 将tuner ID改为"chief"（Keras Tuner要求）
    chief_ip = "0.0.0.0"  # 监听所有IPv4地址
    # 由于chief的计算负载较轻，可以在同一台机器上同时运行一个worker来优化资源利用
    # 取消注释以下代码可在chief机器上启动额外的worker进程
    # import subprocess
    # import sys
    # tf_config["task"]["type"] = "workerX"  # chief机器上的worker
    # os.environ["TF_CONFIG"] = json.dumps(tf_config)
    # subprocess.Popen([sys.executable] + sys.argv,
    #                  stdout=sys.stdout, stderr=sys.stderr)

os.environ["KERASTUNER_TUNER_ID"] = tuner_id  # 设置Keras Tuner的tuner ID
os.environ["KERASTUNER_ORACLE_IP"] = chief_ip  # 设置Keras Tuner的oracle（调度器）IP地址
os.environ["KERASTUNER_ORACLE_PORT"] = chief_port  # 设置Keras Tuner的oracle端口

from pathlib import Path  # 导入Path类
import keras_tuner as kt  # 导入Keras Tuner库
import tensorflow as tf  # 导入TensorFlow

gcs_path = "/gcs/my_bucket/my_hp_search"  # GCS共享目录路径（请替换为你的存储桶名称）

def build_model(hp):  # 定义模型构建函数，hp是超参数对象
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)  # 隐藏层数：0-8
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=256)  # 神经元数：16-256
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2,  # 学习率：1e-4到1e-2
                             sampling="log")  # 对数采样
    optimizer = hp.Choice("optimizer", values=["sgd", "adam"])  # 优化器选择
    if optimizer == "sgd":  # 根据选择创建对应的优化器
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model = tf.keras.Sequential()  # 构建顺序模型
    model.add(tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8))  # 展平层
    for _ in range(n_hidden):  # 添加隐藏层
        model.add(tf.keras.layers.Dense(n_neurons, activation="relu"))  # 全连接层
    model.add(tf.keras.layers.Dense(10, activation="softmax"))  # 输出层
    model.compile(loss="sparse_categorical_crossentropy",  # 编译模型
                  optimizer=optimizer,
                  metrics=["accuracy"])
    return model

hyperband_tuner = kt.Hyperband(  # 创建Hyperband调优器
    build_model, objective="val_accuracy", seed=42,  # 目标：最大化验证准确率
    max_epochs=10, factor=3, hyperband_iterations=2,  # Hyperband参数
    distribution_strategy=tf.distribute.MirroredStrategy(),  # 每个worker使用镜像策略（利用所有GPU）
    directory=gcs_path, project_name="mnist")  # 在GCS共享目录中存储试验结果

# 加载并拆分MNIST数据集
mnist = tf.keras.datasets.mnist.load_data()  # 加载数据集
(X_train_full, y_train_full), (X_test, y_test) = mnist  # 解包
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]  # 拆分验证集
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]  # 拆分标签

tensorboard_log_dir = os.environ["AIP_TENSORBOARD_LOG_DIR"] + "/" + tuner_id  # 每个tuner使用独立的日志目录
tensorboard_cb = tf.keras.callbacks.TensorBoard(tensorboard_log_dir)  # TensorBoard回调
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=5)  # 早停回调
hyperband_tuner.search(X_train, y_train, epochs=10,  # 开始超参数搜索
                       validation_data=(X_valid, y_valid),
                       callbacks=[tensorboard_cb, early_stopping_cb])

if tuner_id == "chief":  # 只有chief保存最终的最佳模型
    best_hp = hyperband_tuner.get_best_hyperparameters()[0]  # 获取最佳超参数
    best_model = hyperband_tuner.hypermodel.build(best_hp)  # 用最佳超参数构建模型
    best_model.save(os.getenv("AIP_MODEL_DIR"), save_format="tf")  # 保存最佳模型到Vertex AI指定路径

Writing my_keras_tuner_search.py


Note that Vertex AI automatically mounts the `/gcs` directory to GCS, using the open source [GCS Fuse adapter](https://cloud.google.com/storage/docs/gcs-fuse). This gives us a shared directory across the workers and the chief, which is required by Keras Tuner. Also note that we set the distribution strategy to a `MirroredStrategy`. This will allow each worker to use all the GPUs on its machine, if there's more than one.


Replace `/gcs/my_bucket/` with <code>/gcs/<i>{bucket_name}</i>/</code>:

In [ ]:
with open("my_keras_tuner_search.py") as f:  # 读取刚才写入的脚本文件
    script = f.read()

with open("my_keras_tuner_search.py", "w") as f:  # 重新写入文件
    f.write(script.replace("/gcs/my_bucket/", f"/gcs/{bucket_name}/"))  # 将脚本中的占位符替换为实际的存储桶名称

Now all we need to do is to start a custom training job based on this script, exactly like in the previous section. Don't forget to add `keras-tuner` to the list of `requirements`:

In [ ]:
hp_search_job = aiplatform.CustomTrainingJob(  # 创建Vertex AI自定义训练任务（用于Keras Tuner超参数搜索）
    display_name="my_hp_search_job",  # 任务显示名称
    script_path="my_keras_tuner_search.py",  # Keras Tuner搜索脚本路径
    container_uri="gcr.io/cloud-aiplatform/training/tf-gpu.2-4:latest",  # 训练容器镜像
    model_serving_container_image_uri=server_image,  # 模型服务容器镜像
    requirements=["keras-tuner~=1.1.2"],  # 依赖：安装keras-tuner
    staging_bucket=f"gs://{bucket_name}/staging",  # 暂存存储桶
)

In [ ]:
mnist_model3 = hp_search_job.run(  # 运行Keras Tuner超参数搜索任务
    machine_type="n1-standard-4",  # 计算实例类型
    replica_count=3,  # 3个副本（1个chief + 2个worker）
    accelerator_type="NVIDIA_TESLA_K80",  # GPU类型
    accelerator_count=2,  # 每个副本2个GPU
)

Training script copied to:
gs://my_bucket/staging/aiplatform-2022-04-15-13:34:32.591-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://my_bucket/staging/aiplatform-custom-training-2022-04-15-13:34:34.453 
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/8601543785521872896?project=522977795627
View backing custom job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5022607048831926272?project=522977795627
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/8601543785521872896 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/8601543785521872896 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/8601543785521872896 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/52297779562

And we have a model!

Let's clean up:

In [ ]:
mnist_model3.delete()  # 删除Vertex AI中的模型
hp_search_job.delete()  # 删除超参数搜索训练任务
blobs = bucket.list_blobs(prefix=f"gs://{bucket_name}/staging/")  # 列出暂存目录中的所有blob
for blob in blobs:  # 遍历每个blob
    blob.delete()  # 删除blob（清理暂存文件）

# Extra Material – Using AutoML to Train a Model

Let's start by exporting the MNIST dataset to PNG images, and prepare an `import.csv` pointing to each image, and indicating the split (training, validation, or test) and the label:

In [ ]:
import matplotlib.pyplot as plt  # 导入matplotlib用于保存图像

mnist_path = Path("datasets/mnist")  # 定义MNIST PNG图片的本地保存路径
mnist_path.mkdir(parents=True, exist_ok=True)  # 创建目录（包括父目录）
idx = 0  # 初始化图片索引
with open(mnist_path / "import.csv", "w") as import_csv:  # 创建CSV导入文件
    for split, X, y in zip(("training", "validation", "test"),  # 遍历三个数据集分割
                           (X_train, X_valid, X_test),
                           (y_train, y_valid, y_test)):
        for image, label in zip(X, y):  # 遍历每张图片和对应标签
            print(f"\r{idx + 1}/70000", end="")  # 打印进度
            filename = f"{idx:05d}.png"  # 生成5位数字的文件名
            plt.imsave(mnist_path / filename, np.tile(image, 3))  # 将灰度图复制3次（RGB）并保存为PNG
            line = f"{split},gs://{bucket_name}/mnist/{filename},{label}\n"  # 构建CSV行：分割类型,GCS路径,标签
            import_csv.write(line)  # 写入CSV文件
            idx += 1  # 索引递增

70000/70000

Let's upload this dataset to GCS:

In [ ]:
upload_directory(bucket, mnist_path)  # 将MNIST PNG图片和CSV文件上传到GCS存储桶

Uploaded datasets/mnist                                              


Now let's create a managed image dataset on Vertex AI:

In [ ]:
from aiplatform.schema.dataset.ioformat.image import single_label_classification  # 导入单标签分类的数据格式schema

mnist_dataset = aiplatform.ImageDataset.create(  # 在Vertex AI上创建托管图像数据集
    display_name="mnist-dataset",  # 数据集显示名称
    gcs_source=[f"gs://{bucket_name}/mnist/import.csv"],  # GCS上的CSV导入文件路径
    project=project_id,  # GCP项目ID
    import_schema_uri=single_label_classification,  # 数据格式schema：单标签图像分类
    sync=True,  # 同步等待数据集创建完成
)

Creating ImageDataset
Create ImageDataset backing LRO: projects/522977795627/locations/us-central1/datasets/7532459492777132032/operations/3812233931370004480
ImageDataset created. Resource name: projects/522977795627/locations/us-central1/datasets/7532459492777132032
To use this ImageDataset in another session:
ds = aiplatform.ImageDataset('projects/522977795627/locations/us-central1/datasets/7532459492777132032')
Importing ImageDataset data: projects/522977795627/locations/us-central1/datasets/7532459492777132032
Import ImageDataset data backing LRO: projects/522977795627/locations/us-central1/datasets/7532459492777132032/operations/3010593197698056192
ImageDataset data imported. Resource name: projects/522977795627/locations/us-central1/datasets/7532459492777132032


Create an AutoML training job on this dataset:

**TODO**

# Exercise Solutions

## 1. to 8.

1. A SavedModel contains a TensorFlow model, including its architecture (a computation graph) and its weights. It is stored as a directory containing a _saved_model.pb_ file, which defines the computation graph (represented as a serialized protocol buffer), and a _variables_ subdirectory containing the variable values. For models containing a large number of weights, these variable values may be split across multiple files. A SavedModel also includes an _assets_ subdirectory that may contain additional data, such as vocabulary files, class names, or some example instances for this model. To be more accurate, a SavedModel can contain one or more _metagraphs_. A metagraph is a computation graph plus some function signature definitions (including their input and output names, types, and shapes). Each metagraph is identified by a set of tags. To inspect a SavedModel, you can use the command-line tool `saved_model_cli` or just load it using `tf.saved_model.load()` and inspect it in Python.
2. TF Serving allows you to deploy multiple TensorFlow models (or multiple versions of the same model) and make them accessible to all your applications easily via a REST API or a gRPC API. Using your models directly in your applications would make it harder to deploy a new version of a model across all applications. Implementing your own microservice to wrap a TF model would require extra work, and it would be hard to match TF Serving's features. TF Serving has many features: it can monitor a directory and autodeploy the models that are placed there, and you won't have to change or even restart any of your applications to benefit from the new model versions; it's fast, well tested, and scales very well; and it supports A/B testing of experimental models and deploying a new model version to just a subset of your users (in this case the model is called a _canary_). TF Serving is also capable of grouping individual requests into batches to run them jointly on the GPU. To deploy TF Serving, you can install it from source, but it is much simpler to install it using a Docker image. To deploy a cluster of TF Serving Docker images, you can use an orchestration tool such as Kubernetes, or use a fully hosted solution such as Google Vertex AI.
3. To deploy a model across multiple TF Serving instances, all you need to do is configure these TF Serving instances to monitor the same _models_ directory, and then export your new model as a SavedModel into a subdirectory.
4. The gRPC API is more efficient than the REST API. However, its client libraries are not as widely available, and if you activate compression when using the REST API, you can get almost the same performance. So, the gRPC API is most useful when you need the highest possible performance and the clients are not limited to the REST API.
5. To reduce a model's size so it can run on a mobile or embedded device, TFLite uses several techniques:
    * It provides a converter which can optimize a SavedModel: it shrinks the model and reduces its latency. To do this, it prunes all the operations that are not needed to make predictions (such as training operations), and it optimizes and fuses operations whenever possible.
    * The converter can also perform post-training quantization: this technique dramatically reduces the model’s size, so it’s much faster to download and store.
    * It saves the optimized model using the FlatBuffer format, which can be loaded to RAM directly, without parsing. This reduces the loading time and memory footprint.
6. Quantization-aware training consists in adding fake quantization operations to the model during training. This allows the model to learn to ignore the quantization noise; the final weights will be more robust to quantization.
7. Model parallelism means chopping your model into multiple parts and running them in parallel across multiple devices, hopefully speeding up the model during training or inference. Data parallelism means creating multiple exact replicas of your model and deploying them across multiple devices. At each iteration during training, each replica is given a different batch of data, and it computes the gradients of the loss with regard to the model parameters. In synchronous data parallelism, the gradients from all replicas are then aggregated and the optimizer performs a Gradient Descent step. The parameters may be centralized (e.g., on parameter servers) or replicated across all replicas and kept in sync using AllReduce. In asynchronous data parallelism, the parameters are centralized and the replicas run independently from each other, each updating the central parameters directly at the end of each training iteration, without having to wait for the other replicas. To speed up training, data parallelism turns out to work better than model parallelism, in general. This is mostly because it requires less communication across devices. Moreover, it is much easier to implement, and it works the same way for any model, whereas model parallelism requires analyzing the model to determine the best way to chop it into pieces. That said, research in this domain is making quick progress (e.g., PipeDream or Pathways), so a mix of model parallelism and data parallelism is probably the way forward.
8. When training a model across multiple servers, you can use the following distribution strategies:
    * The `MultiWorkerMirroredStrategy` performs mirrored data parallelism. The model is replicated across all available servers and devices, and each replica gets a different batch of data at each training iteration and computes its own gradients. The mean of the gradients is computed and shared across all replicas using a distributed AllReduce implementation (NCCL by default), and all replicas perform the same Gradient Descent step. This strategy is the simplest to use since all servers and devices are treated in exactly the same way, and it performs fairly well. In general, you should use this strategy. Its main limitation is that it requires the model to fit in RAM on every replica.
    * The `ParameterServerStrategy` performs asynchronous data parallelism. The model is replicated across all devices on all workers, and the parameters are sharded across all parameter servers. Each worker has its own training loop, running asynchronously with the other workers; at each training iteration, each worker gets its own batch of data and fetches the latest version of the model parameters from the parameter servers, then it computes the gradients of the loss with regard to these parameters, and it sends them to the parameter servers. Lastly, the parameter servers perform a Gradient Descent step using these gradients. This strategy is generally slower than the previous strategy, and a bit harder to deploy, since it requires managing parameter servers. However, it can be useful in some situations, especially when you can take advantage of the asynchronous updates, for example to reduce I/O bottlenecks. This depends on many factors, including hardware, network topology, number of servers, model size, and more, so your mileage may vary.

## 9.
_Exercise: Train a model (any model you like) and deploy it to TF Serving or Google Vertex AI. Write the client code to query it using the REST API or the gRPC API. Update the model and deploy the new version. Your client code will now query the new version. Roll back to the first version._

Please follow the steps in the <a href="#Deploying-TensorFlow-models-to-TensorFlow-Serving-(TFS)">Deploying TensorFlow models to TensorFlow Serving</a> section above.

# 10.
_Exercise: Train any model across multiple GPUs on the same machine using the `MirroredStrategy` (if you do not have access to GPUs, you can use Colaboratory with a GPU Runtime and create two virtual GPUs). Train the model again using the `CentralStorageStrategy `and compare the training time._

Please follow the steps in the [Distributed Training](#Distributed-Training) section above.

# 11.
_Exercise: Train a small model on Google Vertex AI, using TensorFlow Cloud Tuner for hyperparameter tuning._

Please follow the instructions in the _Hyperparameter Tuning using TensorFlow Cloud Tuner_ section in the book.

# Congratulations!

You've reached the end of the book! I hope you found it useful. 😊